# models

> URL bundle and reorder utility functions for sortable queues.

In [ ]:
#| default_exp models

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import dataclass
from typing import List, Callable

In [ ]:
#| export
@dataclass
class SortableQueueUrls:
    """URL endpoints for sortable queue HTMX operations."""
    reorder: str  # POST — Sortable.js drag-end reorder
    remove: str   # POST — Remove item from queue
    clear: str    # POST — Clear all items

## Reorder Utilities

In [ ]:
#| export
def reorder_by_keys(
    items: List[dict],  # Current item list
    new_key_order: List[str],  # Keys in desired order (from Sortable.js form data)
    key_fn: Callable[[dict], str],  # Function to extract key from item
) -> List[dict]:  # Reordered item list
    """Reorder items to match the key order from Sortable.js form data."""
    if not new_key_order:
        return list(items)
    lookup = {key_fn(item): item for item in items}
    reordered = [lookup[k] for k in new_key_order if k in lookup]
    # Append any items not in the new order (safety fallback)
    new_order_set = set(new_key_order)
    for item in items:
        if key_fn(item) not in new_order_set:
            reordered.append(item)
    return reordered

In [ ]:
#| export
def reorder_by_direction(
    items: List[dict],  # Current item list
    item_key: str,  # Key of item to move
    direction: str,  # "up" or "down"
    key_fn: Callable[[dict], str],  # Function to extract key from item
) -> List[dict]:  # Reordered item list
    """Move an item up or down by swapping with its neighbor."""
    result = list(items)
    idx = next((i for i, item in enumerate(result) if key_fn(item) == item_key), None)
    if idx is None:
        return result
    if direction == "up" and idx > 0:
        result[idx], result[idx - 1] = result[idx - 1], result[idx]
    elif direction == "down" and idx < len(result) - 1:
        result[idx], result[idx + 1] = result[idx + 1], result[idx]
    return result

## Tests

In [ ]:
# URL bundle
urls = SortableQueueUrls(reorder="/queue/reorder", remove="/queue/remove", clear="/queue/clear")
assert urls.reorder == "/queue/reorder"
assert urls.remove == "/queue/remove"
assert urls.clear == "/queue/clear"

# Test data
key_fn = lambda item: item["id"]
items = [{"id": "a", "name": "Alpha"}, {"id": "b", "name": "Beta"}, {"id": "c", "name": "Charlie"}]

# reorder_by_keys — normal reorder
result = reorder_by_keys(items, ["c", "a", "b"], key_fn)
assert [key_fn(i) for i in result] == ["c", "a", "b"]

# reorder_by_keys — empty order returns copy
result = reorder_by_keys(items, [], key_fn)
assert [key_fn(i) for i in result] == ["a", "b", "c"]

# reorder_by_keys — missing key in order appends at end
result = reorder_by_keys(items, ["b", "c"], key_fn)
assert [key_fn(i) for i in result] == ["b", "c", "a"]

# reorder_by_keys — extra key in order is ignored
result = reorder_by_keys(items, ["c", "x", "a", "b"], key_fn)
assert [key_fn(i) for i in result] == ["c", "a", "b"]

# reorder_by_direction — move down
result = reorder_by_direction(items, "a", "down", key_fn)
assert [key_fn(i) for i in result] == ["b", "a", "c"]

# reorder_by_direction — move up
result = reorder_by_direction(items, "c", "up", key_fn)
assert [key_fn(i) for i in result] == ["a", "c", "b"]

# reorder_by_direction — can't move first item up
result = reorder_by_direction(items, "a", "up", key_fn)
assert [key_fn(i) for i in result] == ["a", "b", "c"]

# reorder_by_direction — can't move last item down
result = reorder_by_direction(items, "c", "down", key_fn)
assert [key_fn(i) for i in result] == ["a", "b", "c"]

# reorder_by_direction — unknown key returns unchanged
result = reorder_by_direction(items, "x", "down", key_fn)
assert [key_fn(i) for i in result] == ["a", "b", "c"]

print("All models tests passed")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()